# Fase 2 - Classificador Basico de Texto

Este notebook implementa a segunda parte da atividade principal da Fase 2 do CardioIA.

Objetivos desta etapa:
- carregar frases medicas rotuladas;
- transformar texto em vetores numericos com TF-IDF;
- treinar um classificador supervisionado;
- avaliar o desempenho com acuracia e relatorio de classificacao.

## 1. Importacao de bibliotecas

Nesta etapa usamos o `scikit-learn` para vetorizacao textual, treino e avaliacao do modelo.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## 2. Leitura da base rotulada

A base abaixo foi criada para a atividade e contem frases ja classificadas como `alto risco` ou `baixo risco`.

In [ ]:
# Resolve o diretorio base para que o notebook funcione em diferentes contextos de abertura.
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = BASE_DIR / 'data' / 'frases_rotuladas_risco.csv'

# Carrega a base textual com frases e rotulos de risco.
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Separacao entre treino e teste

A base e dividida em dois conjuntos:
- `treino`: usado para aprender o padrao textual;
- `teste`: usado para avaliar o desempenho do classificador.

In [ ]:
# Usa estratificacao para manter o equilibrio entre as classes nos dois conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    df['frase'],
    df['situacao'],
    test_size=0.3,
    random_state=42,
    stratify=df['situacao']
)

print(f'Tamanho de treino: {len(X_train)}')
print(f'Tamanho de teste: {len(X_test)}')

## 4. Vetorizacao com TF-IDF e treino do modelo

O pipeline abaixo faz duas etapas em sequencia:
- converte o texto em vetores numericos com `TfidfVectorizer`;
- aplica `LogisticRegression` como classificador supervisionado.

In [ ]:
pipeline = Pipeline(
    steps=[
        # Gera pesos TF-IDF com unigramas e bigramas para capturar expressoes curtas.
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode')),
        # Usa um modelo simples e adequado para texto em problemas binarios.
        ('modelo', LogisticRegression(max_iter=1000))
    ]
)

# Ajusta o modelo com os dados de treino e gera previsoes no conjunto de teste.
pipeline.fit(X_train, y_train)
previsoes = pipeline.predict(X_test)
acuracia = accuracy_score(y_test, previsoes)

print(f'Acuracia: {acuracia:.2%}')
print('\nMatriz de confusao:')
print(confusion_matrix(y_test, previsoes, labels=['alto risco', 'baixo risco']))
print('\nRelatorio:')
print(classification_report(y_test, previsoes))

## 5. Teste com frases novas

Aqui validamos o comportamento do modelo em frases nao vistas anteriormente.

In [ ]:
# Frases extras para simular uma nova triagem textual.
novas_frases = [
    'dor no peito com suor frio e falta de ar',
    'leve cansaco muscular depois de uma caminhada',
    'palpitacoes com tontura e quase desmaio'
]

list(zip(novas_frases, pipeline.predict(novas_frases)))

## 6. Interpretacao final

Este notebook demonstra um fluxo simples e funcional de classificacao textual para triagem de risco.

Pontos fortes:
- usa uma tecnica classica de NLP com TF-IDF;
- apresenta treino, teste e avaliacao do modelo;
- pode ser demonstrado facilmente no video da atividade.

Limitacoes:
- a base simulada e pequena;
- as frases foram construidas academicamente;
- o modelo nao substitui avaliacao clinica real.